# cli

> `sysone train | eval | predict | serve | run | jobs | publish | login | whoami | image | deploy | undeploy | iac`: the library from the command line.

In [ ]:
#| default_exp cli

In [ ]:
#| export
import json, os, sys
from pathlib import Path
from typing import Optional

import typer

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
from typer.testing import CliRunner
import os, tempfile, logging
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

Each command is a few lines over the library, so anything the command line does, a script does the same way. The heavy imports happen inside the commands, so `sysone --help` is instant.

```{mermaid}
flowchart LR
    TR["sysone train"] --> T1["decision_learner (text or multimodal),<br/>lr_find, fit_one_cycle, calibrate, export"]
    EV["sysone eval"] --> T2["Decider.load, then evaluate"]
    PR["sysone predict"] --> T3["Decider.load, then predict"]
    SV["sysone serve"] --> T4["serve_app under uvicorn"]
    RU["sysone run"] --> T5["cloud.remote: here, Colab, Kaggle or SageMaker"]
    JB["sysone jobs list / watch / logs / fetch / stop"] --> T6["cloud.jobs and a job's methods"]
    PB["sysone publish"] --> T7["the Hub with a model card, or Kaggle Models"]
    LG["sysone login / whoami"] --> T8["auth.login, auth.whoami"]
    DP["sysone image / deploy / undeploy / iac"] --> T9["aws.build_image, deploy.deploy, deploy.cloudformation / cdk_app"]
```

In [ ]:
#| export
app = typer.Typer(help="Jevify BERT-like encoders with sysone.", no_args_is_help=True, add_completion=False)
jobs_app = typer.Typer(help="Jobs started with `sysone run`: list, watch, logs, fetch, stop.", no_args_is_help=True)
app.add_typer(jobs_app, name="jobs")

def _data(data:str, valid, calib:float):
    "TypedDecisions from a JSON-lines file, a registry name or a Hub dataset"
    from sysone.data import TypedDecisions
    from sysone.datasets import REGISTRY
    v = float(valid) if valid is not None and valid.replace(".", "", 1).isdigit() else valid
    if Path(data).exists(): return TypedDecisions.from_json(data, valid=v, calib=calib)
    if data in REGISTRY: return TypedDecisions.from_registry(data, **({"valid": v} if v is not None else {}), calib=calib)
    return TypedDecisions.from_hub(data, valid=v, calib=calib)

## train

`sysone train DATA ENCODER` is the ten-line example: data from a JSON-lines file, a registry name or a Hub dataset, a learner for the encoder (the text application's defaults, or the multimodal one's for a multimodal encoder), one-cycle training, calibration, export.

In [ ]:
#| export
@app.command()
def train(data:str = typer.Argument(..., help="a .jsonl file, a registry name (typed_decisions) or a Hub dataset id"),
          encoder:str = typer.Argument("answerdotai/ModernBERT-large", help="encoder id or folder"),
          output:Optional[Path] = typer.Option(None, help="export folder (SYSONE_OUTPUT, else sysone-export)"),
          regime:str = typer.Option("head", "--train", help="head, lora, mica or full"),
          epochs:int = typer.Option(4, help="training epochs"),
          lr:Optional[float] = typer.Option(None, help="head learning rate (the encoder gets a quarter); lr_find's valley if omitted"),
          valid:Optional[str] = typer.Option(None, help="a split name or a fraction of the training cases"),
          calib:float = typer.Option(0.1, help="fraction of training cases held out for calibration"),
          cache:str = typer.Option("auto", help="auto, masks, rows or none"),
          loss:Optional[str] = typer.Option(None, help="soft_ce or rlcd (the data's default if omitted)"),
          preset:Optional[str] = typer.Option(None, help="hardware preset (SYSONE_PRESET or the machine's if omitted)")):
    "Train a decision model and export it"
    from sysone.models import EncoderSpec
    d = _data(data, valid, calib)
    spec = EncoderSpec.from_pretrained(encoder)
    import importlib
    from sysone import multimodal as mm, text
    from sysone.models import MODEL_TYPES
    make = importlib.import_module(MODEL_TYPES[spec.family]).decision_learner if spec.family in MODEL_TYPES else \
           mm.decision_learner if spec.modality == "multimodal" else text.decision_learner      # the application of the model's family
    learn = make(d, encoder, train=regime, cache=None if cache == "none" else cache, loss=loss, preset=preset)
    typer.echo(repr(learn))
    rate = lr if lr is not None else learn.lr_find().valley
    learn.fit_one_cycle(epochs, lr=rate)
    if d.cases.get("calib") is not None: learn.calibrate()
    if d.cases.get("valid") is not None: typer.echo(json.dumps({k: round(v, 4) for k, v in learn.validate().items()}))
    typer.echo(f"exported to {learn.export(output or Path(os.environ.get('SYSONE_OUTPUT', 'sysone-export')))}")

## eval and predict

In [ ]:
#| export
@app.command("eval")
def eval_(model:str = typer.Argument(..., help="an export folder, a Laya checkpoint or a Hub id"),
          data:str = typer.Option("typed_decisions", help="a .jsonl file with gold, a registry name, or a suite (ag_news, banking77, ...)"),
          split:str = typer.Option("test", help="split, for Hub data"),
          n:Optional[int] = typer.Option(None, help="evaluate the first n cases"),
          by:Optional[str] = typer.Option("workflow", help="report accuracy per value of this record field")):
    "Score a model on typed decisions or a zero-shot suite"
    from sysone.inference import Decider
    from sysone.evaluate import evaluate
    from sysone.datasets import SUITES, suite_records
    from sysone.data import read_jsonl
    jev = Decider.load(model)
    if Path(data).exists(): recs = read_jsonl(data)
    elif data in SUITES: recs, by = suite_records(data, n=n), None
    else:
        from datasets import load_dataset
        recs = list(load_dataset("LocalLLaMA/typed-decisions", "all", split=split)) if data == "typed_decisions" else list(load_dataset(data, split=split))
    recs = recs[:n] if n else recs
    res = evaluate(jev, recs, by=by if recs and by in recs[0] else None)
    typer.echo(json.dumps(res, indent=1, default=float))

@app.command()
def predict(model:str = typer.Argument(..., help="an export folder, a Laya checkpoint or a Hub id"),
            questions:Path = typer.Option(..., help="a JSON file of questions, {qid: {type, instructions, criteria}}"),
            state:Optional[str] = typer.Option(None, help="the state: text, JSON, or @file"),
            device:Optional[str] = typer.Option(None, help="cuda, mps or cpu")):
    "Answer typed questions about one state"
    from sysone.inference import Decider
    s = state if state is not None else sys.stdin.read()
    if s.startswith("@"): s = Path(s[1:]).read_text()
    try: s = json.loads(s)
    except json.JSONDecodeError: pass
    typer.echo(json.dumps(Decider.load(model, device=device).predict(s, json.loads(questions.read_text())), indent=1))

## serve

`sysone serve my-jev` runs the FastAPI app of `sysone.inference` (`POST /v1/systemone`, `GET /health`); it needs `sysone[serve]`. For Laya-format text exports `laya[serve]` serves the same route.

In [ ]:
#| export
@app.command()
def serve(model:str = typer.Argument(..., help="an export folder, a Laya checkpoint or a Hub id"),
          host:str = typer.Option("127.0.0.1"), port:int = typer.Option(8000), device:Optional[str] = typer.Option(None)):
    "Serve POST /v1/systemone and GET /health"
    import uvicorn
    from sysone.inference import Decider, serve_app
    uvicorn.run(serve_app(Decider.load(model, device=device)), host=host, port=port)

## run, jobs and publish

In [ ]:
#| export
@app.command()
def run(entry:Path = typer.Argument(..., help="the notebook or script, the same file that runs here"),
        on:str = typer.Option("local", help="local, colab, kaggle or sagemaker"),
        hw:Optional[str] = typer.Option(None, help="cpu, mps, t4, t4x2, l4, a100, h100, tpu, v5e-1, v6e-8, trn1.2xlarge ..."),
        data:Optional[str] = typer.Option(None, help="a local path, hf:owner/name, kaggle:owner/name or s3://bucket/key; the job reads it from SYSONE_DATA"),
        push:Optional[str] = typer.Option(None, help="the Hub repo the export is pushed to"),
        track:bool = typer.Option(False, help="log the training to MLflow (the job's runs come back with fetch --sync)"),
        experiment:Optional[str] = typer.Option(None, help="the MLflow experiment, with --track"),
        max_hours:Optional[float] = typer.Option(None, help="stop the job after this long"),
        env:Optional[list[str]] = typer.Option(None, help="KEY=VALUE for the job (repeatable)"),
        arg:Optional[list[str]] = typer.Option(None, help="an argument for a script (repeatable)"),
        wait:bool = typer.Option(False, help="follow the job until it ends"),
        dry_run:bool = typer.Option(False, "--dry-run", help="write the job's files, submit nothing")):
    "Run a notebook or script as a job, here in the background or on Colab, Kaggle or SageMaker"
    from sysone import cloud
    kv = dict(e.split("=", 1) for e in env or [])
    tr = ({"experiment": experiment} if experiment else True) if track else None
    job = cloud.remote(entry, on=on, hw=hw, data=data, push=push, track=tr, max_hours=max_hours, env=kv, args=arg, dry_run=dry_run)
    typer.echo(json.dumps({k: v for k, v in job.__dict__.items() if k not in ("options", "last")}, indent=1, default=str))
    if wait and not dry_run: job.wait()

@jobs_app.command("list")
def jobs_list(backend:Optional[str] = typer.Option(None, help="only this backend's jobs"),
              active:bool = typer.Option(False, help="only jobs still running")):
    "List jobs with their latest progress"
    from sysone import cloud
    typer.echo(repr(cloud.jobs(backend=backend, active=active)))

@jobs_app.command("watch")
def jobs_watch(every:float = typer.Option(30, help="seconds between refreshes")):
    "Refresh the table of running jobs until they end"
    from sysone import cloud
    cloud.watch(every)

@jobs_app.command("logs")
def jobs_logs(job:str = typer.Argument(..., help="a job id, or a unique prefix of one")):
    "Print a job's log"
    from sysone import cloud
    j = cloud.get_job(job); j.refresh(); typer.echo(j.logs())

@jobs_app.command("fetch")
def jobs_fetch(job:str = typer.Argument(..., help="a job id, or a unique prefix of one"),
               dest:Optional[Path] = typer.Argument(None, help="where the outputs go (the job's folder by default)"),
               sync:bool = typer.Option(False, help="copy the job's MLflow runs into your store")):
    "Bring a finished job's outputs here; prints the export's folder"
    from sysone import cloud
    typer.echo(str(cloud.get_job(job).fetch(dest, sync=sync)))

@jobs_app.command("stop")
def jobs_stop(job:str = typer.Argument(..., help="a job id, or a unique prefix of one")):
    "Stop a job"
    from sysone import cloud
    typer.echo(cloud.get_job(job).stop().status)

@app.command()
def publish(model:Path = typer.Argument(..., help="an export folder"),
            to:str = typer.Option("hub", help="hub or kaggle"),
            repo:Optional[str] = typer.Option(None, help="Hub repo id (to hub)"),
            tag:Optional[str] = typer.Option(None, help="a tag for this commit, e.g. v1 (to hub)"),
            revision:Optional[str] = typer.Option(None, help="a branch to push to instead of main (to hub)"),
            license:Optional[str] = typer.Option(None, help="the license in the model card, e.g. apache-2.0 (to hub)"),
            card:bool = typer.Option(True, help="write the model card as README.md (to hub)"),
            owner:Optional[str] = typer.Option(None, help="Kaggle owner (to kaggle)"),
            name:Optional[str] = typer.Option(None, help="Kaggle model slug (to kaggle)"),
            private:bool = typer.Option(True), dry_run:bool = typer.Option(False, "--dry-run")):
    "Publish an export to the Hub (the default, with a model card) or to Kaggle Models"
    if to == "kaggle":
        from sysone.kaggle import publish_kaggle
        typer.echo(json.dumps(publish_kaggle(model, owner, name or model.name, private=private, dry_run=dry_run)))
    elif to == "hub":
        if repo is None: raise typer.BadParameter("--repo is needed to publish to the Hub")
        from sysone.hub import push_to_hub
        res = push_to_hub(model, repo, private=private, card=card, license=license, revision=revision, tag=tag, dry_run=dry_run)
        typer.echo(json.dumps({k: v for k, v in res.items() if k != "card"}, indent=1))
    else: raise typer.BadParameter(f"unknown destination {to!r}")

@app.command()
def login(service:str = typer.Argument("hf", help="hf, kaggle, colab or aws"),
          local:bool = typer.Option(False, help="aws: print the settings for a local emulator (floci), to eval in a shell"),
          profile:Optional[str] = typer.Option(None, help="aws: the profile to use"),
          full:bool = typer.Option(False, help="colab: all the CLI's permissions (Google Cloud, Drive), for colab auth and drivemount")):
    "Log in to a service: the browser sign-ins print a URL (Kaggle and Colab then ask for the code it shows)"
    from sysone import auth
    if service == "aws" and local:
        for k, v in auth.aws_local().items(): typer.echo(f"export {k}={v}")
        return
    kw = {"profile": profile} if service == "aws" else {"full": full} if service == "colab" else {}
    typer.echo(f"{service}: {auth.login(service, **kw)}")

@app.command()
def whoami():
    "The account each service uses on this machine"
    from sysone import auth
    for k, v in auth.whoami().items(): typer.echo(f"{k:7s} {v}")

@app.command()
def image(gpu:bool = typer.Option(False, help="the CUDA image (PyTorch's) instead of the CPU one"),
          platform:Optional[str] = typer.Option(None, help="e.g. linux/amd64, for AWS from an Apple machine"),
          tag:Optional[str] = typer.Option(None, help="the image's tag (sysone:VERSION-cpu by default)")):
    "Build sysone's image, which trains SageMaker jobs (`train`) and serves endpoints (`serve`)"
    from sysone.aws import build_image
    typer.echo(build_image(tag, gpu=gpu, platform=platform))

def _backend(on:str, endpoint_url:Optional[str]):
    from sysone.cloud import Local, SageMaker
    return Local() if on == "local" else SageMaker(endpoint_url=endpoint_url)

@app.command()
def deploy(model:Path = typer.Argument(..., help="an export folder"),
           on:str = typer.Option("local", help="local (Docker here) or sagemaker"),
           hw:Optional[str] = typer.Option(None, help="the instance's hardware: cpu, t4, l4, inf2.xlarge ..."),
           name:Optional[str] = typer.Option(None, help="the endpoint's name"),
           endpoint_url:Optional[str] = typer.Option(None, help="sagemaker: an emulator's endpoint, e.g. http://localhost:4566 for floci"),
           serverless_mb:Optional[int] = typer.Option(None, help="sagemaker: a serverless endpoint with this much memory")):
    "Serve an export: in Docker here, or as a SageMaker endpoint; prints the endpoint"
    from sysone.deploy import deploy as dep
    kw = {"serverless": {"MemorySizeInMB": serverless_mb, "MaxConcurrency": 5}} if serverless_mb else {}
    ep = dep(model, on=_backend(on, endpoint_url), hw=hw, name=name, **kw)
    typer.echo(json.dumps({"name": ep.name, "url": getattr(ep, "url", None), "container": getattr(ep, "container", None)}))

@app.command()
def undeploy(name:str = typer.Argument(..., help="the endpoint's name"),
             on:str = typer.Option("local", help="local or sagemaker"),
             endpoint_url:Optional[str] = typer.Option(None, help="sagemaker: an emulator's endpoint")):
    "Remove an endpoint `sysone deploy` made"
    from sysone.deploy import LocalEndpoint, SageMakerEndpoint
    ep = LocalEndpoint(name, f"sysone-endpoint-{name}"[:63], "") if on == "local" else SageMakerEndpoint(name, _backend(on, endpoint_url))
    ep.delete(); typer.echo(f"removed {name}")

@app.command()
def iac(name:str = typer.Argument(..., help="the endpoint's name"),
        out:Path = typer.Option(Path("infra"), help="the folder to write to"),
        kind:str = typer.Option("cloudformation", help="cloudformation or cdk"),
        instance:str = typer.Option("ml.m5.xlarge", help="the instance type"),
        autoscale:Optional[str] = typer.Option(None, help="MIN,MAX,INVOCATIONS_PER_INSTANCE"),
        mlflow:bool = typer.Option(False, help="add a SageMaker MLflow tracking server"),
        image_uri:str = typer.Option("<account>.dkr.ecr.<region>.amazonaws.com/sysone:latest", help="cdk: the image"),
        model_data:str = typer.Option("s3://<bucket>/sysone/models/<name>/model.tar.gz", help="cdk: the model's S3 URI")):
    "Write the infrastructure of a sysone endpoint as a CloudFormation template or a CDK app"
    from sysone.deploy import cloudformation, write_template, cdk_app
    sc = tuple(float(x) if i == 2 else int(x) for i, x in enumerate(autoscale.split(","))) if autoscale else None
    if kind == "cdk": typer.echo(str(cdk_app(out, name, image_uri, model_data, instance=instance, autoscale=sc, mlflow=mlflow)))
    else: typer.echo(str(write_template(cloudformation(name, instance=instance, autoscale=sc, mlflow=mlflow), out / f"{name}.yaml")))

def main(): app()

## Trying it

On the fixtures, end to end: train, predict, evaluate, and a dry run of a cloud job:

In [ ]:
runner = CliRunner()
r = runner.invoke(app, ["--help"])
test_eq(r.exit_code, 0)
print(r.output)

                                                                                
 Usage: root [OPTIONS] COMMAND [ARGS]...                                        
                                                                                
 Jevify BERT-like encoders with sysone.                                         
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --help          Show this message and exit.                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ train    Train a decision model and export it                                │
│ eval     Score a model on typed decisions or a zero-shot suite               │
│ predict  Answer typed questions about one state                              │
│ serve    Serve POST /v1/sy

In [ ]:
out = Path(tempfile.mkdtemp()) / "cli-jev"
os.environ["SYSONE_CACHE"] = tempfile.mkdtemp()
r = runner.invoke(app, ["train", "fixtures/typed_decisions_tiny.jsonl", "fixtures/tiny-modernbert", "--output", str(out),
                        "--epochs", "2", "--lr", "3e-3", "--valid", "0.25", "--preset", "cpu"])
if r.exit_code != 0: print(r.output); raise r.exception
test_eq((out / "sysone.json").exists(), True)
print(r.output[-400:])

_per_second': '1977', 'train_steps_per_second': '273.7', 'train_loss': '0.9173', 'epoch': '2'}
{"accuracy": 0.36, "soft_accuracy": 0.3767, "brier": 0.26, "kl": 0.4434, "tv": 0.366, "ece": 0.1159, "score_mae": 0.8379, "within_one": 0.6, "n": 25, "accuracy_choice": 0.4286, "accuracy_score": 0.3, "accuracy_noul": 0.375}
exported to /var/folders/ym/y7_gxdb90s312g8bz06h3qqm0000gn/T/tmpcjz36ll1/cli-jev



In [ ]:
qf = Path(tempfile.mkdtemp()) / "q.json"
qf.write_text(json.dumps({"urgent": {"type": "noul", "instructions": "This needs attention today."}}))
r = runner.invoke(app, ["predict", str(out), "--questions", str(qf), "--state", "The server room is flooding.", "--device", "cpu"])
test_eq(r.exit_code, 0)
test_eq(set(json.loads(r.output)["urgent"]), {"type", "noul", "confidence", "answer_confidence"})

In [ ]:
r = runner.invoke(app, ["eval", str(out), "--data", "fixtures/typed_decisions_tiny.jsonl", "--n", "4"])
test_eq((r.exit_code, json.loads(r.output)["n"]), (0, 20))

In [ ]:
os.environ["SYSONE_JOBS"] = str(Path(tempfile.mkdtemp()) / "jobs.json")
os.environ["SYSONE_JOBS_DIR"] = tempfile.mkdtemp()
script = Path(tempfile.mkdtemp()) / "train.py"; script.write_text("print('hi')\n")
r = runner.invoke(app, ["run", str(script), "--on", "kaggle", "--hw", "t4x2", "--push", "me/jev", "--dry-run"])
test_eq((r.exit_code, json.loads(r.output)["machine"], json.loads(r.output)["status"]), (0, "NvidiaTeslaT4", "dry-run"))
r = runner.invoke(app, ["run", str(script), "--on", "colab", "--hw", "a100", "--dry-run", "--env", "A=1"])
test_eq(json.loads(r.output)["machine"], "--gpu A100")
r = runner.invoke(app, ["run", str(script), "--on", "colab", "--hw", "t4x2", "--dry-run"])
test_eq((r.exit_code, "one GPU" in str(r.exception)), (1, True))
test_eq(runner.invoke(app, ["jobs", "list"]).exit_code, 0)
r = runner.invoke(app, ["publish", str(out), "--dry-run", "--repo", "me/jev", "--tag", "v1"])
pub = json.loads(r.output)
test_eq((pub["repo"], pub["private"], pub["tag"], "README.md" in pub["files"]), ("me/jev", True, "v1", True))
test_eq(runner.invoke(app, ["login", "aws", "--local"]).output.splitlines()[0], "export AWS_ENDPOINT_URL=http://localhost:4566")
infra = Path(tempfile.mkdtemp())
r = runner.invoke(app, ["iac", "my-jev", "--out", str(infra), "--autoscale", "1,4,100", "--mlflow"])
test_eq((r.exit_code, Path(r.output.strip()).name), (0, "my-jev.yaml"))
r = runner.invoke(app, ["iac", "my-jev", "--out", str(infra / "cdk"), "--kind", "cdk"])
test_eq(sorted(p.name for p in Path(r.output.strip()).iterdir()), ["app.py", "cdk.json", "requirements.txt", "template.json"])
r = runner.invoke(app, ["publish", str(out), "--to", "kaggle", "--owner", "me", "--name", "my-jev", "--dry-run"])
test_eq((r.exit_code, json.loads(r.output)["variation"][:4]), (0, ["kaggle", "models", "variations", "create"]))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()